# Data Wrangling with DataFrames Coding Quiz

Use this Jupyter notebook to find the answers to the quiz in the previous section. There is an answer key in the next part of the lesson.

In [2]:
from pyspark.sql import SparkSession

# TODOS: 
# 1) import any other libraries you might need
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
from pprint import pprint

# 2) instantiate a Spark session
spark = SparkSession \
    .builder \
    .appName("Wrangling Data") \
    .getOrCreate()
# 3) read in the data set located at the path "data/sparkify_log_small.json"
path = "data/sparkify_log_small.json"
user_log = spark.read.json(path)

# 4) write code to answer the quiz questions
user_log.head()

Row(artist='Showaddywaddy', auth='Logged In', firstName='Kenneth', gender='M', itemInSession=112, lastName='Matthews', length=232.93342, level='paid', location='Charlotte-Concord-Gastonia, NC-SC', method='PUT', page='NextSong', registration=1509380319284, sessionId=5132, song='Christmas Tears Will Fall', status=200, ts=1513720872284, userAgent='"Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.125 Safari/537.36"', userId='1046')

In [4]:
user_log.printSchema()

root
 |-- artist: string (nullable = true)
 |-- auth: string (nullable = true)
 |-- firstName: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- itemInSession: long (nullable = true)
 |-- lastName: string (nullable = true)
 |-- length: double (nullable = true)
 |-- level: string (nullable = true)
 |-- location: string (nullable = true)
 |-- method: string (nullable = true)
 |-- page: string (nullable = true)
 |-- registration: long (nullable = true)
 |-- sessionId: long (nullable = true)
 |-- song: string (nullable = true)
 |-- status: long (nullable = true)
 |-- ts: long (nullable = true)
 |-- userAgent: string (nullable = true)
 |-- userId: string (nullable = true)



# Question 1

Which page did user id "" (empty string) NOT visit?

In [26]:
# TODO: write your code to answer question 1
all_pages = user_log.select('page').dropDuplicates().sort('page')
this_user = user_log.filter("userId == ''").select('page').dropDuplicates().sort('page')

all_pages.show()
this_user.show()

# not_visited:
set(all_pages.toPandas().page).difference(this_user.toPandas().page)

+----------------+
|            page|
+----------------+
|           About|
|       Downgrade|
|           Error|
|            Help|
|            Home|
|           Login|
|          Logout|
|        NextSong|
|   Save Settings|
|        Settings|
|Submit Downgrade|
|  Submit Upgrade|
|         Upgrade|
+----------------+

+-----+
| page|
+-----+
|About|
| Help|
| Home|
|Login|
+-----+



{'Downgrade',
 'Error',
 'Logout',
 'NextSong',
 'Save Settings',
 'Settings',
 'Submit Downgrade',
 'Submit Upgrade',
 'Upgrade'}

# Question 2 - Reflect

What type of user does the empty string user id most likely refer to?


In [2]:
# TODO: use this space to explore the behavior of the user with an empty string


# Question 3

How many female users do we have in the data set?

In [27]:
# TODO: write your code to answer question 3
user_log.filter("gender == 'F'").select('userID').dropDuplicates().count()

462

# Question 4

How many songs were played from the most played artist?

In [56]:
# TODO: write your code to answer question 4
from pyspark.sql.functions import col

user_log.filter("page=='NextSong'").groupBy("artist").count().sort("count", ascending=False).show()
user_log.filter( col('artist')=='Coldplay' ).groupby('song').count().sort('count', ascending=False).show(53)

+--------------------+-----+
|              artist|count|
+--------------------+-----+
|            Coldplay|   83|
|       Kings Of Leon|   69|
|Florence + The Ma...|   52|
|            BjÃÂ¶rk|   46|
|       Dwight Yoakam|   45|
|       Justin Bieber|   43|
|      The Black Keys|   40|
|         OneRepublic|   37|
|                Muse|   36|
|        Jack Johnson|   36|
|           Radiohead|   31|
|        Taylor Swift|   29|
|               Train|   28|
|Barry Tuckwell/Ac...|   28|
|          Lily Allen|   28|
|          Nickelback|   27|
|           Daft Punk|   27|
|           Metallica|   27|
|          Kanye West|   26|
|Red Hot Chili Pep...|   24|
+--------------------+-----+
only showing top 20 rows

+--------------------+-----+
|                song|count|
+--------------------+-----+
|              Yellow|   21|
|              Clocks|   12|
|       The Scientist|   11|
|             Fix You|    8|
|              Shiver|    8|
|         In My Place|    4|
|             Tro

# Question 5 (challenge)

How many songs do users listen to on average between visiting our home page? Please round your answer to the closest integer.



In [77]:
# TODO: write your code to answer question 5
from pyspark.sql import Window
df = user_log.filter(col('page').isin(['Home','NextSong'])).select(['page','userID','ts', 'sessionID']).sort('ts')
df.groupby('sessionID').count().agg({"count":"avg"}).show()

+-----------------+
|       avg(count)|
+-----------------+
|6.422372881355932|
+-----------------+



In [83]:
# TODO: filter out 0 sum and max sum to get more exact answer
from pyspark.sql.functions import udf, desc
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import sum as Fsum

function = udf(lambda ishome : int(ishome == 'Home'), IntegerType())

user_window = Window \
    .partitionBy('userID') \
    .orderBy(desc('ts')) \
    .rangeBetween(Window.unboundedPreceding, 0)

cusum = df.filter((df.page == 'NextSong') | (df.page == 'Home')) \
    .select('userID', 'page', 'ts') \
    .withColumn('homevisit', function(col('page'))) \
    .withColumn('period', Fsum('homevisit').over(user_window))

cusum.filter((cusum.page == 'NextSong')) \
    .groupBy('userID', 'period') \
    .agg({'period':'count'}) \
    .agg({'count(period)':'avg'}).show()

+------------------+
|avg(count(period))|
+------------------+
| 6.898347107438017|
+------------------+

